In [10]:
from lxml import etree

input_file = r"C:\Users\zcohe\Jmod\JMod_Profiling\Output\Line_Profiler_MS1_Cor_Channels\14_14.5_2025-05-12_5plex_200pg_20win_24nce_1.mzML"
output_file = r"C:\Users\zcohe\Jmod\JMod_Profiling\Output\Line_Profiler_MS1_Cor_Channels\14_14.5_2025-05-12_5plex_200pg_20win_24nce_1.mzML"

rt_min = 14.0  # minutes
rt_max = 14.5  # minutes

tree = etree.parse(input_file)
root = tree.getroot()

ns = {"mzml": "http://psi.hupo.org/ms/mzml"}  # mzML namespace

# Get spectrumList node
spectrum_list = root.xpath(".//mzml:spectrumList", namespaces=ns)[0]

removed = 0
for spectrum in spectrum_list.xpath("mzml:spectrum", namespaces=ns):
    # Look for scan start time cvParam
    rt_elem = spectrum.xpath('.//mzml:cvParam[@accession="MS:1000016"]', namespaces=ns)
    if rt_elem:
        rt = float(rt_elem[0].get("value"))
        units = rt_elem[0].get("unitName", "minute")
        if units.startswith("second"):  # convert if seconds
            rt /= 60.0
        if rt < rt_min or rt > rt_max:
            spectrum_list.remove(spectrum)
            removed += 1

# Update the count attribute
new_count = len(spectrum_list.xpath("mzml:spectrum", namespaces=ns))
spectrum_list.set("count", str(new_count))

print(f"Removed {removed} spectra, kept {new_count}")

# Save filtered mzML
tree.write(output_file, pretty_print=True, xml_declaration=True, encoding="UTF-8")


Removed 38698 spectra, kept 501


In [14]:
from pyteomics import mzml

output_file = r"C:\Users\zcohe\Jmod\JMod_Profiling\Output\Line_Profiler_MS1_Cor_Channels\14_14.5_2025-05-12_5plex_200pg_20win_24nce_1.mzML"

first_time = None
last_time = None
ms1_count = 0
ms2_count = 0

with mzml.MzML(output_file) as reader:
    for spec in reader:
        ms_level = spec['ms level']
        rt = spec['scanList']['scan'][0]['scan start time']

        if ms_level == 1:
            ms1_count += 1
            if first_time is None:
                first_time = rt
            last_time = rt
        elif ms_level == 2:
            ms2_count += 1

if first_time is not None and last_time is not None:
    gradient_length = last_time - first_time
    print(last_time)
    print(first_time)
    print(f"Gradient length: {gradient_length:.2f} min")
    print(f"MS1 scans: {ms1_count}")
    print(f"MS2 scans: {ms2_count}")
else:
    print("No MS1 scans found in the file.")

14.494016883333
14.001494616667
Gradient length: 0.49 min
MS1 scans: 44
MS2 scans: 457
